# ACTUNEO AI: ask questions in plain words

In this notebook you type a question, a local language model works out **which ACTUNEO function** answers it and **with what numbers**, and ACTUNEO does the calculation.

The model never does the arithmetic. It only chooses a function and fills in its arguments, so the figures come from the library. What the model can get wrong is the choice of function or an argument, which is why every answer below shows both. Check them before you trust the result.

**Before you start**, install a model from a terminal (not from here, so you can answer the question and see the download progress):

```
actuneo-ai models
actuneo-ai setup --force
```

Then run the cells below from top to bottom.

## 1. Load the model

The first cell shows which model is installed. Loading takes a few seconds the first time.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import actuneo.ai as ai

ai.status()
model = ai.get_backend()

## 2. The functions the model can choose from

Eleven ACTUNEO calculations, each with a plain description of what it does and what it needs. The model sees the descriptions; Python runs the functions.

The mortality tables are South African (SA85-90 for assured lives, SAIML98 and SAIFL98 for annuitants), because those cover the full range of ages. Change them in this cell if you want another basis.

In [ ]:
from actuneo import zimbabwe
from actuneo.finance import Bond, Loan
from actuneo.life import LifeAssurance
from actuneo.loss_reserving import ChainLadder, MackChainLadder, load_genins, load_raa
from actuneo.mortality import MortalityTable
from actuneo.pensions import DCProjection, pension_annuity

ASSURED = MortalityTable.from_south_africa("sa8590")                  # ages 15 to 104
ANNUITANTS = {"M": MortalityTable.from_south_africa("saiml98"),       # ages 40 to 110
              "F": MortalityTable.from_south_africa("saifl98")}
TRIANGLES = {"raa": load_raa, "genins": load_genins}


def _triangle(name):
    key = str(name).lower().replace("triangle", "").strip()
    if key not in TRIANGLES:
        raise ValueError(f"unknown triangle '{name}'. Available: RAA, GenIns")
    return TRIANGLES[key]()


def life_expectancy(age, sex="M"):
    table = ANNUITANTS[sex] if age >= 40 else ASSURED
    return f"{table.life_expectancy(age):.1f} more years ({table.name})"


def survival_probability(age, years):
    return f"{ASSURED.npx(age, years):.1%} chance of surviving {years} years from age {age} (SA85-90)"


def annuity_value(age, annual_amount, interest_rate, sex="M"):
    factor = pension_annuity(ANNUITANTS[sex], age, interest_rate)
    return f"{annual_amount * factor:,.0f} (annuity factor {factor:.3f}, paid monthly in advance)"


def term_assurance_premium(age, term, sum_assured, interest_rate):
    rate = LifeAssurance(ASSURED, interest_rate=interest_rate).net_annual_premium(age, term, "term")
    return f"{sum_assured * rate:,.2f} a year (net premium, no expenses, SA85-90)"


def chain_ladder_reserve(triangle):
    return f"IBNR {ChainLadder(_triangle(triangle)).total_ibnr:,.0f}"


def mack_standard_error(triangle):
    mack = MackChainLadder(_triangle(triangle), est_sigma="mack")
    return (f"standard error {mack.total_mack_se:,.0f} on a reserve of {mack.total_ibnr:,.0f} "
            f"({mack.total_mack_se / mack.total_ibnr:.0%})")


def loan_repayment(amount, interest_rate, years):
    return f"{Loan(amount, interest_rate, years, payments_per_year=12).payment:,.2f} a month"


def bond_price(coupon_rate, term, yield_rate):
    return f"{Bond(coupon_rate, term, frequency=2).price(yield_rate):.2f} per 100 nominal"


def present_value(amount, years, interest_rate):
    return f"{amount / (1 + interest_rate) ** years:,.2f}"


def retirement_fund_projection(age, retirement_age, salary, contribution_rate, investment_return):
    fund = DCProjection(age, retirement_age, salary, member_rate=contribution_rate,
                        investment_return=investment_return).fund_at_retirement
    return f"{fund:,.0f} at age {retirement_age} (level salary, no charges)"


def zwl_to_zig(amount):
    return f"ZiG {zimbabwe.zwl_to_zwg(amount):,.2f}"


FUNCTIONS = {f.__name__: f for f in [
    life_expectancy, survival_probability, annuity_value, term_assurance_premium,
    chain_ladder_reserve, mack_standard_error, loan_repayment, bond_price, present_value,
    retirement_fund_projection, zwl_to_zig]}

RATE = {"type": "number", "description": "as a decimal, for example 0.05 for 5%"}
SEX = {"type": "string", "enum": ["M", "F"], "description": "M for a man, F for a woman"}


def tool(name, description, required, **properties):
    return {"name": name, "description": description,
            "parameters": {"type": "object", "properties": properties, "required": required}}


TOOLS = [
    tool("life_expectancy", "Expectation of life: how many more years a person of an age is expected to live",
         ["age"], age={"type": "integer"}, sex=SEX),
    tool("survival_probability", "Probability or chance that a person of an age survives, or lives, a number of years",
         ["age", "years"], age={"type": "integer"},
         years={"type": "integer", "description": "number of years to survive"}),
    tool("annuity_value", "Value, worth or cost of an annuity or pension of an annual amount paid for life from an age",
         ["age", "annual_amount", "interest_rate"], age={"type": "integer"},
         annual_amount={"type": "number", "description": "amount paid per year"},
         interest_rate=RATE, sex=SEX),
    tool("term_assurance_premium", "Yearly premium or cost of term assurance, life cover of a sum assured for a term of years",
         ["age", "term", "sum_assured", "interest_rate"], age={"type": "integer"},
         term={"type": "integer", "description": "years of cover"},
         sum_assured={"type": "number"}, interest_rate=RATE),
    tool("chain_ladder_reserve", "IBNR claims reserve for a run-off triangle by the chain ladder method",
         ["triangle"], triangle={"type": "string", "enum": ["RAA", "GenIns"]}),
    tool("mack_standard_error", "Mack standard error: the uncertainty of the chain ladder reserve for a triangle",
         ["triangle"], triangle={"type": "string", "enum": ["RAA", "GenIns"]}),
    tool("loan_repayment", "Monthly repayment on a loan or mortgage of an amount borrowed over a number of years",
         ["amount", "interest_rate", "years"], amount={"type": "number"}, interest_rate=RATE,
         years={"type": "integer"}),
    tool("bond_price", "Price of a bond from its coupon rate, its term in years and the yield",
         ["coupon_rate", "term", "yield_rate"], coupon_rate=RATE, term={"type": "integer"},
         yield_rate=RATE),
    tool("present_value", "Present value, what a single amount due or payable in a number of years is worth today",
         ["amount", "years", "interest_rate"], amount={"type": "number"},
         years={"type": "integer"}, interest_rate=RATE),
    tool("retirement_fund_projection", "Size of a retirement fund or pot at retirement from salary, contributions and investment return",
         ["age", "retirement_age", "salary", "contribution_rate", "investment_return"],
         age={"type": "integer"}, retirement_age={"type": "integer"},
         salary={"type": "number", "description": "salary per year"},
         contribution_rate={"type": "number", "description": "total contributions as a decimal share of salary"},
         investment_return=RATE),
    tool("zwl_to_zig", "Convert an amount of Zimbabwe dollars (ZWL) to Zimbabwe Gold (ZiG)",
         ["amount"], amount={"type": "number"}),
]
print(len(TOOLS), "functions ready:", ", ".join(FUNCTIONS))

## 3. Ask

`ask()` sends your question to the model, runs the function it chose, and shows all three: the function, the arguments and ACTUNEO's result.

Two safeguards are built in. Arguments outside a sensible range (an age of 400, a term of a million years) are refused instead of calculated. And when no function fits, nothing is calculated and the model's own words are not shown, because a small model will state false things with confidence.

**Read the function line before the result.** A wrong function with sensible-looking arguments still gives a number.

In [ ]:
# Arguments outside these ranges are refused instead of being calculated, so that a
# misread question cannot turn into a believable number.
LIMITS = {"age": (0, 110), "retirement_age": (40, 80), "years": (1, 100), "term": (1, 100),
          "interest_rate": (0, 0.5), "yield_rate": (0, 0.5), "coupon_rate": (0, 0.5),
          "investment_return": (0, 0.5), "contribution_rate": (0, 1)}


def check_arguments(arguments):
    for name, value in arguments.items():
        if name in LIMITS and not LIMITS[name][0] <= value <= LIMITS[name][1]:
            raise ValueError(f"{name}={value!r} is outside the range {LIMITS[name]}. "
                             "The model probably misread the question.")
        if isinstance(value, (int, float)) and not isinstance(value, bool) and value < 0:
            raise ValueError(f"{name}={value!r} is negative")


def ask(question, show=True, show_model_text=False):
    """Let the model choose a function, then let ACTUNEO calculate."""
    call = model.call_tools(question, TOOLS)
    if call.name is None:
        if show:
            print(f"Q: {question}")
            print("   No ACTUNEO function fits this question, so nothing was calculated.")
            if show_model_text:
                print("   The model's own words (NOT from ACTUNEO, may be wrong):")
                print(f"   {call.text[:400]}")
            print()
        return call, None
    try:
        check_arguments(call.arguments)
        answer = FUNCTIONS[call.name](**call.arguments)
    except Exception as problem:
        answer = f"not calculated: {problem}"
    if show:
        arguments = ", ".join(f"{key}={value!r}" for key, value in call.arguments.items())
        print(f"Q: {question}")
        print(f"   function : {call.name}({arguments})")
        print(f"   ACTUNEO  : {answer}\n")
    return call, answer


ask("How long is a 65 year old man expected to live?");

### Simple questions

One clear request each, with the numbers stated outright. A model that suits this job should get all of these.

In [ ]:
for question in [
    "What is the life expectancy of a 45 year old woman?",
    "What is the probability that a 30 year old survives 20 years?",
    "Value a pension of 6000 a year from age 65 at 5% interest.",
    "Premium for a 20-year term assurance of 100000 for a 35 year old at 6% interest",
    "Estimate the IBNR for the RAA triangle with the chain ladder method.",
    "What is the Mack standard error for the GenIns triangle?",
    "What is the monthly repayment on a loan of 50000 over 10 years at 8% interest?",
    "Price a 10-year bond with a 6% coupon at a yield of 7%.",
    "What is the present value of 1000 due in 5 years at 10% interest?",
    "Convert 1,000,000 ZWL to ZiG.",
]:
    ask(question)

### Your own question

Change the text and run the cell. Things to try: say it the way you would say it to a colleague, leave out the word the function uses, or give the rate in words.

In [ ]:
ask("A woman aged 60 wants 12,000 a year for life. What does that cost at 4%?");

## 4. Test it properly

Each test below is a question with the function and arguments that a correct reading would give. `run_tests` marks each one:

- **ok**: right function and right arguments
- **arg**: right function, but an argument is wrong or missing
- **NO**: wrong function, or a function chosen when none fits

Only the arguments listed are checked, so an extra one the model adds does not count against it.

In [ ]:
from actuneo.ai.benchmark import arguments_match


def run_tests(cases, title=""):
    rows = []
    for question, expected_function, expected_arguments in cases:
        call, answer = ask(question, show=False)
        function_ok = call.name == expected_function
        arguments_ok = function_ok and arguments_match(expected_arguments, call.arguments)
        mark = "ok " if arguments_ok else ("arg" if function_ok else "NO ")
        print(f"[{mark}] {question}")
        if not arguments_ok:
            print(f"        expected: {expected_function} {expected_arguments}")
            print(f"        got     : {call.name} {call.arguments}")
        rows.append({"question": question, "expected": expected_function, "chosen": call.name,
                     "arguments": call.arguments, "function_ok": function_ok,
                     "arguments_ok": arguments_ok, "result": answer})
    table = pd.DataFrame(rows)
    print(f"\n{title} {int(table['arguments_ok'].sum())} of {len(table)} fully right, "
          f"{int(table['function_ok'].sum())} of {len(table)} with the right function")
    return table

### Level 1: stated plainly

In [ ]:
LEVEL_1 = [
    ("What is the life expectancy of a 45 year old woman?", "life_expectancy", {"age": 45, "sex": "F"}),
    ("How long is a man aged 70 expected to live?", "life_expectancy", {"age": 70, "sex": "M"}),
    ("What is the probability that a 30 year old survives 20 years?", "survival_probability", {"age": 30, "years": 20}),
    ("Value a pension of 6000 a year from age 65 at 5% interest.", "annuity_value", {"age": 65, "annual_amount": 6000, "interest_rate": 0.05}),
    ("Premium for a 20-year term assurance of 100000 for a 35 year old at 6% interest", "term_assurance_premium", {"age": 35, "term": 20, "sum_assured": 100000, "interest_rate": 0.06}),
    ("Estimate the IBNR for the RAA triangle with the chain ladder method.", "chain_ladder_reserve", {"triangle": "RAA"}),
    ("What is the Mack standard error for the GenIns triangle?", "mack_standard_error", {"triangle": "GenIns"}),
    ("What is the monthly repayment on a loan of 50000 over 10 years at 8% interest?", "loan_repayment", {"amount": 50000, "interest_rate": 0.08, "years": 10}),
    ("Price a 10-year bond with a 6% coupon at a yield of 7%.", "bond_price", {"coupon_rate": 0.06, "term": 10, "yield_rate": 0.07}),
    ("What is the present value of 1000 due in 5 years at 10% interest?", "present_value", {"amount": 1000, "years": 5, "interest_rate": 0.10}),
    ("Convert 1,000,000 ZWL to ZiG.", "zwl_to_zig", {"amount": 1000000}),
    ("I am 30 and retire at 60 on a salary of 12000, contributing 10% with a 7% investment return. How big is my retirement fund?", "retirement_fund_projection", {"age": 30, "retirement_age": 60, "salary": 12000, "contribution_rate": 0.10, "investment_return": 0.07}),
]
level_1 = run_tests(LEVEL_1, "Level 1:")

### Level 2: said the way people say it

Different words from the function's description, numbers written as `120k` or in words, and details the model has to pick out of a sentence.

In [ ]:
LEVEL_2 = [
    ("My client is a 52-year-old lady. How many more years can she expect?", "life_expectancy", {"age": 52, "sex": "F"}),
    ("Chance a 50 year old lives another 10 years?", "survival_probability", {"age": 50, "years": 10}),
    ("What does it cost to give a man of 60 an income of 12,000 a year for the rest of his life, at 4%?", "annuity_value", {"age": 60, "annual_amount": 12000, "interest_rate": 0.04, "sex": "M"}),
    ("How much a year for 250,000 of life cover over 25 years for someone aged 40? Use 5% interest.", "term_assurance_premium", {"age": 40, "term": 25, "sum_assured": 250000, "interest_rate": 0.05}),
    ("What's the IBNR on GenIns?", "chain_ladder_reserve", {"triangle": "GenIns"}),
    ("How uncertain is the chain ladder reserve for RAA?", "mack_standard_error", {"triangle": "RAA"}),
    ("Borrowing 120k over 15 years at eight and a half percent. What do I pay each month?", "loan_repayment", {"amount": 120000, "interest_rate": 0.085, "years": 15}),
    ("A 15-year bond pays a 9% coupon and yields are 11%. What should it trade at?", "bond_price", {"coupon_rate": 0.09, "term": 15, "yield_rate": 0.11}),
    ("What is 25,000 payable in 12 years worth today at 7.5%?", "present_value", {"amount": 25000, "years": 12, "interest_rate": 0.075}),
    ("How much is ZWL 5 million in ZiG?", "zwl_to_zig", {"amount": 5000000}),
]
level_2 = run_tests(LEVEL_2, "Level 2:")

### Level 3: advanced

These need the model to do some working before it can fill in the arguments, or to recognise that nothing fits:

- **arithmetic first**: "alive at 70" from age 45 is 25 years; 750 a month is 9,000 a year; 7.5% plus 10% is 17.5%
- **a distraction**: a detail in the question that is not an argument
- **a trap**: words from one function used about something else
- **missing information**: the right reply is that it cannot be answered, not a guess
- **not a calculation**: general questions should be left alone

Small models usually fail the arithmetic ones. That is expected: it shows where the size of model matters.

In [ ]:
LEVEL_3 = [
    # arithmetic first
    ("If I am 45 now, what are the odds I am still alive at 70?", "survival_probability", {"age": 45, "years": 25}),
    ("A widow aged 63 will receive 750 a month for life. What is that worth at 6%?", "annuity_value", {"age": 63, "annual_amount": 9000, "interest_rate": 0.06, "sex": "F"}),
    ("I am 28, earn 18,000 a year and retire at 60. I put in 7.5% and my employer adds 10%. Returns are 8%. How big is my pot?", "retirement_fund_projection", {"age": 28, "retirement_age": 60, "salary": 18000, "contribution_rate": 0.175, "investment_return": 0.08}),
    ("A bond issued 5 years ago with a 20-year term pays 8%. Yields are 6%. What is the price now?", "bond_price", {"coupon_rate": 0.08, "term": 15, "yield_rate": 0.06}),
    # a distraction
    ("Our Harare branch lent 80,000 to a 40 year old teacher over 20 years at 12%. Monthly instalment?", "loan_repayment", {"amount": 80000, "interest_rate": 0.12, "years": 20}),
    ("The RAA triangle has 10 accident years. What reserve does chain ladder give?", "chain_ladder_reserve", {"triangle": "RAA"}),
    # a trap
    ("What is the life expectancy of a 10-year bond?", None, {}),
    ("What is the probability that it rains in Bulawayo tomorrow?", None, {}),
    # missing information
    ("What is the monthly repayment on my loan?", None, {}),
    ("Value my pension.", None, {}),
    # not a calculation
    ("Explain what IBNR means.", None, {}),
    ("Which is better, chain ladder or Bornhuetter-Ferguson?", None, {}),
    ("Good morning, can you help me?", None, {}),
]
level_3 = run_tests(LEVEL_3, "Level 3:")

### The score

Write these three numbers down for each model you try.

In [ ]:
scores = pd.DataFrame({
    "questions": [len(t) for t in (level_1, level_2, level_3)],
    "right function": [int(t["function_ok"].sum()) for t in (level_1, level_2, level_3)],
    "fully right": [int(t["arguments_ok"].sum()) for t in (level_1, level_2, level_3)],
}, index=["Level 1: stated plainly", "Level 2: natural wording", "Level 3: advanced"])
scores.loc["Total"] = scores.sum()
print("Model:", ai.status(quiet=True)["local_model"]["name"])
scores

In [ ]:
# Every question that was not fully right, with what the model chose
missed = pd.concat([level_1, level_2, level_3])
missed.loc[~missed["arguments_ok"], ["question", "expected", "chosen", "arguments"]]

## 5. What it cannot do

Run these to see the limits for yourself. `show_model_text=True` prints what the model says when no function fits.

- **Two things at once.** The model picks one function per question, so a comparison gets no answer or half of one.
- **A basis you did not state.** Tables and conventions are fixed in section 2. The model will not ask which table you meant.
- **Explanations.** The model's own words are not ACTUNEO output. A small model will invent definitions, figures and sources. With Qwen3 0.6B, "Explain what IBNR means" came back as "International Business and Risk Management". Do not rely on that text.

In [ ]:
ask("Compare the chain ladder reserves for RAA and GenIns.", show_model_text=True)
ask("What is the life expectancy of a 30 year old on the Zimbabwe 2023 tables?", show_model_text=True)
ask("Explain what IBNR means.", show_model_text=True);

## 6. Compare models

To try a bigger model, install it in a terminal and run this notebook again from the top (restart the kernel first):

```
actuneo-ai models
actuneo-ai setup --model small --force
```

Keys: `minimal` (Qwen3 0.6B), `tiny` (1.7B), `small` (4B), `default` (8B), `strong` (14B), `max` (30B-A3B), and the challengers `phi4-mini`, `gemma3-4b`, `phi4-mini-reasoning`, `deepseek-r1-14b`.

The library also has its own fixed benchmark of twenty requests, which does not depend on this notebook's functions:

In [ ]:
# results = ai.benchmark()

## 7. Add your own function

Write the Python function, add it to `FUNCTIONS`, and describe it in `TOOLS` in the words a user would use. The description matters more than anything else: the model chooses on it, and only the five functions whose descriptions share most words with the question are shown to the model.

In [ ]:
def future_value(amount, years, interest_rate):
    return f"{amount * (1 + interest_rate) ** years:,.2f}"


FUNCTIONS["future_value"] = future_value
TOOLS.append(tool("future_value", "Future value: what an amount invested today grows or accumulates to after a number of years",
                  ["amount", "years", "interest_rate"], amount={"type": "number"},
                  years={"type": "integer"}, interest_rate=RATE))

ask("If I invest 5,000 today at 9%, what will it grow to in 12 years?");